# Notebook 08 — P3 chọn hệ thống; P4 suy luận chạy riêng

P3 chỉ dùng OOF đã lưu, chạy CPU. So 6 chiến lược, xem từng fold/từng lớp và lưu recipe có hash nguồn.
Mỗi fold được chấm sau khi thành phần/trọng số/bias đều được fit trên 4 fold còn lại.
Đây là cross-fitting tầng quyết định trên OOF phát triển, **không phải nested CV độc lập toàn pipeline**.
Việc chọn chiến lược cuối cũng dùng các điểm phát triển này; chưa có điểm test độc lập cho lựa chọn cuối.

Chạy các mục 1–5 để xem hoặc tái tạo P3. Mục P4 bên dưới mặc định tắt để Run All không tự train.


## 1. Thiết lập CPU


In [1]:
import json
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/sqe").exists())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sqe.data import TASKS, load_en
from sqe.decision import apply_bias
from sqe.metrics import fold_scores
from sqe.submission import read_submission, write_submission
RUNS_DIR = ROOT / "outputs/runs"
SUB = ROOT / "outputs/submissions"
df = load_en()
pd.set_option("display.precision", 4)


## 2. Tạo lại hoặc đọc kết quả P3

`RECOMPUTE_P3=False`: đọc kết quả đã kiểm chứng. Đổi True chỉ khi muốn chạy lại phép chọn bằng CPU.
Candidate raw loại random/majority, bản TF-IDF trùng, OOF đã chỉnh bias và OOF hỏng. Qwen chỉ có nhãn fold 0 nên không vào ensemble xác suất.


In [2]:
RECOMPUTE_P3 = False
if RECOMPUTE_P3:
    import runpy
    runpy.run_path(str(ROOT / "scripts/run_p3_selection.py"), run_name="__main__")
manifest = json.loads((ROOT / "outputs/selection/manifest.json").read_text(encoding="utf-8"))
print("Scope:", manifest["evaluation_scope"])
print("Bị loại:", manifest["rejected_candidates"])
display(pd.read_csv(ROOT / "outputs/results/p3_candidates.csv"))


Scope: development_oof_decision_crossfit
Bị loại: [{'subtask': 'A', 'model': 'deberta_comment', 'reason': 'deberta_comment/A: Nonfinite probabilities (NaN/Inf); failed run cannot be scored.'}, {'subtask': 'B', 'model': 'deberta_comment', 'reason': 'deberta_comment/B: Nonfinite probabilities (NaN/Inf); failed run cannot be scored.'}]


,subtask,model,macro_f1,f1_no,f1_yes,macro_f1_std,f1_yes_implicit,f1_yes_explicit
0,A,mt_twhate_comment,0.7354,0.7653,0.7055,0.0111,NaN,NaN
1,A,mtcw_twhate_comment,0.7301,0.7556,0.7045,0.0127,NaN,NaN
2,A,roberta_comment,0.7274,0.7590,0.6958,0.0098,NaN,NaN
3,A,roberta_concat_tdc,0.7113,0.7494,0.6732,0.0162,NaN,NaN
4,A,roberta_pair_title,0.7266,0.7618,0.6915,0.0120,NaN,NaN
5,A,roberta_pair_title_desc,0.7288,0.7627,0.6949,0.0053,NaN,NaN
6,A,tfidf_comment,0.7002,0.7333,0.6671,0.0209,NaN,NaN
7,A,tfidf_comment_title,0.6775,0.7124,0.6426,0.0234,NaN,NaN
8,A,twhate_comment,0.7335,0.7649,0.7020,0.0115,NaN,NaN
9,B,mt_twhate_comment,0.6006,0.7501,NaN,0.0259,0.4313,0.6204


## 3. So 6 chiến lược

Best single chọn một model; top-3 mean trung bình ba model; greedy chọn thêm model khi điểm fit tăng.
Mỗi cách có bản cộng bias. Xem macro-F1, std, từng lớp và từng fold. Số fold thắng chỉ là mô tả, không chứng minh ý nghĩa thống kê.


In [3]:
strategies = pd.read_csv(ROOT / "outputs/results/p3_strategies.csv")
folds = pd.read_csv(ROOT / "outputs/results/p3_folds.csv")
display(strategies)
display(folds)


,subtask,strategy,macro_f1,f1_no,f1_yes,macro_f1_std,delta_vs_best_single,wins_vs_best_single,evaluation_scope,f1_yes_implicit,f1_yes_explicit,selected
0,A,best single,0.7323,0.7640,0.7007,0.0146,0.0000,0,development_oof_decision_crossfit,NaN,NaN,False
1,A,best single + bias,0.7338,0.7688,0.6988,0.0143,0.0014,3,development_oof_decision_crossfit,NaN,NaN,False
2,A,top-3 mean,0.7419,0.7711,0.7126,0.0102,0.0095,4,development_oof_decision_crossfit,NaN,NaN,False
3,A,top-3 mean + bias,0.7396,0.7625,0.7166,0.0095,0.0072,3,development_oof_decision_crossfit,NaN,NaN,False
4,A,greedy,0.7455,0.7766,0.7144,0.0142,0.0131,4,development_oof_decision_crossfit,NaN,NaN,False
5,A,greedy + bias,0.7460,0.7720,0.7200,0.0103,0.0137,4,development_oof_decision_crossfit,NaN,NaN,True
6,B,best single,0.6006,0.7501,NaN,0.0259,0.0000,0,development_oof_decision_crossfit,0.4313,0.6204,False
7,B,best single + bias,0.5962,0.7339,NaN,0.0160,-0.0044,2,development_oof_decision_crossfit,0.4412,0.6136,False
8,B,top-3 mean,0.6043,0.7417,NaN,0.0164,0.0037,2,development_oof_decision_crossfit,0.4540,0.6173,False
9,B,top-3 mean + bias,0.6075,0.7391,NaN,0.0127,0.0068,3,development_oof_decision_crossfit,0.4631,0.6202,True


,fold,macro_f1,f1_no,f1_yes,subtask,strategy,delta_vs_best_single,f1_yes_implicit,f1_yes_explicit
0,0,0.7431,0.7628,0.7233,A,best single,0.0000,NaN,NaN
1,1,0.7462,0.7754,0.7170,A,best single,0.0000,NaN,NaN
2,2,0.7432,0.7862,0.7002,A,best single,0.0000,NaN,NaN
3,3,0.7115,0.7511,0.6718,A,best single,0.0000,NaN,NaN
4,4,0.7178,0.7443,0.6913,A,best single,0.0000,NaN,NaN
5,0,0.7422,0.7671,0.7172,A,best single + bias,-0.0009,NaN,NaN
6,1,0.7488,0.7811,0.7165,A,best single + bias,0.0026,NaN,NaN
7,2,0.7441,0.7938,0.6944,A,best single + bias,0.0009,NaN,NaN
8,3,0.7115,0.7511,0.6718,A,best single + bias,0.0000,NaN,NaN
9,4,0.7224,0.7508,0.6940,A,best single + bias,0.0046,NaN,NaN


## 4. Chốt recipe

Quy tắc đã đặt trước: chọn ensemble chỉ khi hơn best single (có/không bias) ít nhất 0,005 macro-F1.
Sau chọn chiến lược, fit thành phần/trọng số/bias trên toàn bộ OOF để dùng ở P4. Điểm cross-fit phía trên đánh giá các lượt fit theo fold, không phải điểm test của recipe toàn OOF.


In [4]:
RECIPE = json.loads((ROOT / "outputs/submissions/recipe.json").read_text(encoding="utf-8"))
for t in "AB":
    r = RECIPE[t]
    print(t, r["strategy"], r["components"], "bias:", r["bias"])
    print("development macro-F1:", r["development_macro_f1"], "rule:", r["selection_rule"])


A greedy + bias {'mt_twhate_comment': 1.0, 'mtcw_twhate_comment': 1.0, 'roberta_pair_title': 1.0, 'roberta_pair_title_desc': 1.0, 'tfidf_comment': 4.0, 'tfidf_comment_title': 2.0} bias: [0.0, 0.0]
development macro-F1: 0.7460196470173812 rule: {'best_scoring_strategy': 'greedy + bias', 'best_simple_strategy': 'best single + bias', 'gain_over_simple': 0.012238806849993011, 'minimum_ensemble_gain': 0.005, 'rule': 'Choose best ensemble only if gain over best single (with/without bias) >= 0.005'}
B top-3 mean + bias {'mt_twhate_comment': 1.0, 'mtcw_twhate_comment': 1.0, 'twhate_comment': 1.0} bias: [0.0, 0.2, 0.1]
development macro-F1: 0.6074550860206245 rule: {'best_scoring_strategy': 'top-3 mean + bias', 'best_simple_strategy': 'best single', 'gain_over_simple': 0.0068392962899589405, 'minimum_ensemble_gain': 0.005, 'rule': 'Choose best ensemble only if gain over best single (with/without bias) >= 0.005'}


## 5. Bảng báo cáo và kết luận

Giữ cột evaluation_scope: Qwen chỉ fold 0, supervised là 5-fold, P3 là development decision cross-fit. Không xếp hạng trực tiếp các scope khác nhau.


In [5]:
display(pd.read_csv(ROOT / "outputs/results/summary_table.csv"))


,notebook,model,input,subtask,date,macro_f1,f1_no,f1_yes,macro_f1_std,note,...,f1_yes_explicit,experiment,evaluation_scope,n_eval,run_fingerprint,failed_attempts,status,source_oof,source_oof_sha256,stage
0,3,majority,comment,A,2026-10-11,0.3575,0.7150,0.0000,2.5410e-04,baseline cổ điển (cấu hình notebook 02),...,NaN,NaN,cv5_video,NaN,NaN,NaN,completed_valid,NaN,NaN,majority
1,3,tfidf_lr,comment,A,2026-10-11,0.7002,0.7333,0.6671,2.0919e-02,baseline cổ điển (cấu hình notebook 02),...,NaN,NaN,cv5_video,NaN,NaN,NaN,completed_valid,NaN,NaN,TF-IDF
2,3,roberta-base,comment,A,2026-10-11,0.7274,0.7590,0.6958,9.7725e-03,lr2e-5 ep4 bs16 len128,...,NaN,NaN,cv5_video,NaN,NaN,NaN,completed_valid,NaN,NaN,RoBERTa comment
3,4,roberta-base,pair_title_desc,A,2026-10-13,0.7288,0.7627,0.6949,5.2874e-03,"C2, len256",...,NaN,NaN,cv5_video,NaN,NaN,NaN,completed_valid,NaN,NaN,best context tried
4,4,cardiffnlp/twitter-roberta-base-hate-latest,comment,A,2026-10-13,0.7335,0.7649,0.7020,1.1522e-02,"đổi encoder, len128",...,NaN,NaN,cv5_video,NaN,NaN,NaN,completed_valid,NaN,NaN,best encoder tried
5,5,cardiffnlp/twitter-roberta-base-hate-latest,comment,A,2026-10-13,0.7362,0.7708,0.7016,1.1014e-02,MT + bias; development OOF cross-fit,...,NaN,mt_twhate_comment_bias,cv5_video,2989.0,NaN,NaN,completed_valid,mt_twhate_comment_bias_A.csv,f8b44b41829b4fe2d2f453181b180728f624e13d31b0cc...,best 05 variant tried
6,6,qwen2.5:7b-instruct-q4_K_M,title+desc+comment,A,2026-10-14T06:07:03.755531+00:00,0.6927,0.7384,0.6471,NaN,few_shot; CHỈ fold 0; Q4_K_M; không phải CV 5 ...,...,NaN,few_shot,fold0,599.0,92c7563e09fa4fdf6db88be9574b2723eef8437a85145b...,0.0,completed_valid,NaN,NaN,Qwen fold 0 only
7,8,final: greedy + bias,mt_twhate_comment+mtcw_twhate_comment+roberta_...,A,2026-10-18,0.7460,0.7720,0.7200,1.0278e-02,P3: jointly fit component selection and bias o...,...,NaN,selected_recipe,development_oof_decision_crossfit,2989.0,2904d685b24102f4afd58c611f779fddb074d2402ed9f1...,NaN,completed_valid,outputs/selection/selected_A.csv,92f63029c5fd67f6e303cb4b90ecaa4245ec832c35153f...,selected P3 strategy
8,3,majority,comment,B,2026-10-11,0.2134,0.6401,NaN,1.4778e-05,baseline cổ điển (cấu hình notebook 02),...,0.0000,NaN,cv5_video,NaN,NaN,NaN,completed_valid,NaN,NaN,majority
9,3,tfidf_lr,comment,B,2026-10-11,0.5249,0.6782,NaN,1.3831e-02,baseline cổ điển (cấu hình notebook 02),...,0.5107,NaN,cv5_video,NaN,NaN,NaN,completed_valid,NaN,NaN,TF-IDF


P3 hoàn thành khi có bảng 12 dòng/60 kết quả theo fold, recipe kèm hash, giải thích lựa chọn và notebook 07 dùng đúng OOF của chiến lược đã chọn. Xem [P3_REVIEW.md](../outputs/selection/P3_REVIEW.md) và ROADMAP.md.


## P4 — Huấn luyện và thử suy luận (mặc định tắt)

Các hàm cũ dưới đây train lại trong bộ nhớ; **chưa đáp ứng save/reload checkpoint ở phiên mới**.
Cần hoàn thiện lưu/nạp model ở P4 trước khi coi hệ thống sẵn sàng nộp. Lịch và quy cách BTC cần xác minh khi có test thật.


In [6]:
RUN_P4_DRYRUN = False
RUN_OFFICIAL_TEST = False


In [7]:
_cache = {}


def component_cfg(name, t):
    """Đọc cấu hình đã dùng khi chạy CV của thành phần `name` (từ outputs/runs/)."""
    for f in sorted(RUNS_DIR.glob(f"{name}_*.json")):
        if f.stem[len(name) + 1:] in ("A", "B", "AB"):
            meta = json.loads(f.read_text(encoding="utf-8"))
            if t in meta["tasks"]:
                return meta["cfg"], tuple(meta["tasks"])
    raise FileNotFoundError(name)


def predict_component(name, t, df_train, df_test, tag):
    from sqe import classical
    from sqe.train import fit_full_and_predict
    labels = TASKS[t][1]
    if name.startswith("tfidf_"):
        texts = {"tfidf_comment": lambda d: d.yt_comment,
                 "tfidf_comment_title": lambda d: d.yt_comment + "\n" + d.yt_title}[name]
        m = classical.make_model("tfidf_lr").fit(texts(df_train).tolist(), df_train[TASKS[t][0]])
        p = m.predict_proba(texts(df_test).tolist())
        return p[:, [list(m.classes_).index(l) for l in labels]]
    cfg, tasks = component_cfg(name, t)
    key = (tag, name, tasks)
    if key not in _cache:
        print(f"  train {name} {tasks} trên {len(df_train)} mẫu ...")
        _cache[key] = fit_full_and_predict(df_train, df_test, cfg, tasks, log=lambda *_: None)
    return _cache[key][t]


def predict_recipe(df_train, df_test, t, tag):
    labels = TASKS[t][1]
    r = RECIPE[t]
    P = sum(w * predict_component(n, t, df_train, df_test, tag) for n, w in r["components"].items())
    P = apply_bias(P / sum(r["components"].values()), np.array(r["bias"]))
    return np.array(labels)[P.argmax(1)]

In [8]:
if RUN_P4_DRYRUN:
    dry_train = df[df.fold != 0].reset_index(drop=True)
    dry_test = df[df.fold == 0].reset_index(drop=True)
    truth = dry_test[["StereoQueerEval_id", "stereotype", "hate_speech"]].copy()
    dry_test = dry_test.drop(columns=["stereotype", "hate_speech", "target"])
    
    SUB = ROOT / "outputs" / "submissions"
    dry_rows = []
    for t in "AB":
        col, labels = TASKS[t]
        pred = predict_recipe(dry_train, dry_test, t, tag="dry")
        path = write_submission(dry_test.StereoQueerEval_id, pred, t, SUB / f"dryrun_fold0_{t}.tsv")
        back = read_submission(path, t)
        assert list(back.StereoQueerEval_id) == list(truth.StereoQueerEval_id)
        s = f1_score(truth[col], back.label, labels=labels, average="macro")
        nested_f0 = fold_scores(pd.read_csv(ROOT / RECIPE[t]["selected_strategy_oof"]), labels).set_index("fold").macro_f1[0]
        dry_rows.append({"subtask": t, "file": path.name, "dòng": len(back), "macro_f1 fold 0 (chạy thử)": s,
                         "macro_f1 fold 0 (OOF development cross-fit)": nested_f0})
    display(pd.DataFrame(dry_rows))
    print("5 dòng đầu file nộp B:")
    print("".join(open(SUB / "dryrun_fold0_B.tsv", encoding="utf-8").readlines()[:5]))
else:
    print("P4 dry-run tắt: không train trong P3.")


P4 dry-run tắt: không train trong P3.


In [9]:
if RUN_OFFICIAL_TEST:
    TEST_PATH = ROOT.parent / "StereoQueerEval_SemEval-2027" / "StereoQueerEval_EN_test.tsv"   # ← sửa khi có file thật
    
    if TEST_PATH.exists():
        test = pd.read_csv(TEST_PATH, sep="\t", dtype=str, keep_default_na=False, encoding="utf-8")
        assert {"StereoQueerEval_id", "yt_title", "yt_description", "yt_comment"} <= set(test.columns)
        print("Số mẫu test:", len(test), "(guideline: 743)")
        for t in "AB":
            pred = predict_recipe(df, test, t, tag="test")
            p = write_submission(test.StereoQueerEval_id, pred, t, SUB / f"test_EN_{t}.tsv")
            print("Đã ghi", p, "| phân bố nhãn:", pd.Series(pred).value_counts(normalize=True).round(3).to_dict())
    else:
        print(f"Chưa có file test ({TEST_PATH.name}). Bỏ qua — ô này dành cho tháng 01/2027.")
else:
    print("Suy luận test chính thức tắt.")


Suy luận test chính thức tắt.
